# Huberman Lab Youtube Extraction

### Setup

In [1]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time
import pandas as pd

In [21]:
# Get playlist page using Selenium since the content is dynamically loaded
driver = webdriver.Chrome()  # Make sure you have ChromeDriver installed
driver.get("https://www.youtube.com/playlist?list=PLPNW_gerXa4Pc8S2qoUQc5e8Ir97RLuVW")

# Function to scroll to bottom of page
def scroll_to_bottom():
    last_height = driver.execute_script("return document.documentElement.scrollHeight")
    while True:
        # Scroll down
        driver.execute_script("window.scrollTo(0, document.documentElement.scrollHeight);")
        # Wait for new videos to load
        time.sleep(2)
        # Calculate new scroll height
        new_height = driver.execute_script("return document.documentElement.scrollHeight")
        if new_height == last_height:
            break
        last_height = new_height

# Scroll to load all videos
scroll_to_bottom()

# Wait for the video elements to load
wait = WebDriverWait(driver, 30)
video_elements = wait.until(EC.presence_of_all_elements_located((By.CSS_SELECTOR, "ytd-playlist-video-renderer")))

# Extract video information
video_data = []
for element in video_elements:
    # Get URL and title
    video_link = element.find_element(By.CSS_SELECTOR, "a#video-title").get_attribute('href')
    title = element.find_element(By.CSS_SELECTOR, "a#video-title").text
    
    if video_link and '/watch?v=' in video_link:
        video_data.append({
            'url': video_link,
            'title': title
        })

driver.quit()

# Create DataFrame
df_videos = pd.DataFrame(video_data)
df_videos


,url,title
0,https://www.youtube.com/watch?v=q-wRvsiGYIs&li...,"AMA #19: Collagen vs. Whey Protein, Creatine, ..."
1,https://www.youtube.com/watch?v=ssmwxKPFMFU&li...,Protocols to Improve Vision & Eyesight | Huber...
2,https://www.youtube.com/watch?v=J7yn4tJEmJU&li...,Tools for Overcoming Substance & Behavioral Ad...
3,https://www.youtube.com/watch?v=7MEhDlw1e9k&li...,How to Build Endurance | Huberman Lab Essentials
4,https://www.youtube.com/watch?v=UyneMnERmnI&li...,How to Improve Your Vitality & Heal From Disea...


In [24]:
df_videos.to_csv('data/huberman_videos.csv', index=False)